<a href="https://colab.research.google.com/github/samradnyi-AIMLtech/RAG-Based-Document-Intelligence-System/blob/main/Copy_of_RAG_Document_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pypdf
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install transformers
!pip -q install accelerate
!pip -q install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 42.3 MB/s eta 0:00:00


In [3]:
import os
import re
import numpy as np
import faiss
import torch
import gradio as gr

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
# PDF files will be uploaded through the Gradio application.
# We initialize the document storage here.

all_pages = []
chunks = []

print("PDF document system initialized.")

PDF document system initialized.


In [5]:
def extract_text_from_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):

        text = page.extract_text()

        if text:

            pages.append({
                "document": os.path.basename(pdf_path),
                "page": page_number,
                "text": text
            })

    return pages


print("PDF extraction function ready.")

PDF extraction function ready.


In [6]:
def clean_text(text):

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    text = text.strip()

    return text


print("Text cleaning function ready.")

Text cleaning function ready.


In [ ]:
for page in all_pages:

    page["text"] = clean_text(page["text"])

print("Text cleaning completed.")

Text cleaning completed.


In [7]:
def create_chunks(
    pages,
    chunk_size=800,
    overlap=300
):

    chunks = []

    for page in pages:

        text = page["text"]

        page_number = page["page"]

        document_name = page["document"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk_text = text[start:end]

            if chunk_text.strip():

                chunks.append({
                    "text": chunk_text,
                    "page": page_number,
                    "document": document_name
                })

            start += (
                chunk_size - overlap
            )

    return chunks


print("Chunking function ready.")

Chunking function ready.


In [8]:
chunks = create_chunks(
    all_pages,
    chunk_size=800,
    overlap=150
)

print("Total chunks:", len(chunks))

Total chunks: 0


In [9]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [10]:
#Embedding Helper

def create_embeddings(chunks):

    if not chunks:
        raise ValueError(
            "No chunks available. "
            "Upload and process PDFs first."
        )

    chunk_texts = [
        chunk["text"]
        for chunk in chunks
        if chunk["text"].strip()
    ]

    if not chunk_texts:
        raise ValueError(
            "The PDFs contain no extractable text."
        )

    embeddings = embedding_model.encode(
        chunk_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    embeddings = np.asarray(embeddings)

    if embeddings.ndim != 2:
        raise ValueError(
            f"Unexpected embedding shape: {embeddings.shape}"
        )

    return embeddings


print("Embedding function ready.")


Embedding function ready.


In [11]:
# FAISS

def create_faiss_index(embeddings):

    if embeddings is None:
        raise ValueError(
            "Embeddings have not been created."
        )

    if embeddings.ndim != 2:
        raise ValueError(
            f"Invalid embedding shape: {embeddings.shape}"
        )

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(
        dimension
    )

    index.add(
        embeddings.astype("float32")
    )

    return index


print("FAISS function ready.")

FAISS function ready.


In [12]:
def retrieve_documents(
    question,
    selected_documents=None,
    top_k=5
):

    question_embedding = (
        embedding_model.encode(
            [question],
            convert_to_numpy=True
        )
    )

    search_k = min(
        top_k * 5,
        index.ntotal
    )

    distances, indices = index.search(
        question_embedding.astype(
            "float32"
        ),
        search_k
    )

    results = []

    for distance, idx in zip(
        distances[0],
        indices[0]
    ):

        if idx >= len(chunks):
            continue

        chunk = chunks[idx]

        # If documents are selected,
        # only search those PDFs.
        if (
            selected_documents
            and chunk["document"]
            not in selected_documents
        ):
            continue

        results.append({
            "text": chunk["text"],
            "page": chunk["page"],
            "document": chunk["document"],
            "distance": float(distance)
        })

        if len(results) >= top_k:
            break

    return results


print("Multi-document retrieval ready.")

Multi-document retrieval ready.


In [13]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

llm = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("LLM loaded!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded!


In [14]:
def create_prompt(
    question,
    retrieved_documents,
    mode="Ask"
):

    context = ""
    for doc in retrieved_documents:
        context += f"Source ({doc['document']}, p.{doc['page']}): {doc['text']}\n\n"

    # Select brief task instruction based on mode
    if mode == "Summarize":
        instruction = "Summarize the key information from the context."
    elif mode == "Compare":
        instruction = "Compare similarities and differences in the context."
    elif mode == "Extract":
        instruction = "Extract key facts from the context in bullet points."
    elif mode == "Research":
        instruction = "Provide a detailed analysis using the context."
    else:
        instruction = "Answer the question directly using the context."

    prompt = f"""Task: {instruction} If the answer is not in the context, say "I could not find the answer in the uploaded documents."

Context:
{context}

Question: {question}
Answer:"""

    return prompt

In [15]:
def generate_answer(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.2
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [16]:
def ask_documents(
    question,
    selected_documents=None,
    mode="Ask",
    top_k=3
):

    retrieved_documents = (
        retrieve_documents(
            question,
            selected_documents,
            top_k
        )
    )

    if not retrieved_documents:

        return (
            "I could not find relevant "
            "information in the selected "
            "documents.",
            []
        )

    prompt = create_prompt(
        question,
        retrieved_documents,
        mode
    )

    answer = generate_answer(
        prompt
    )

    return (
        answer,
        retrieved_documents
    )

In [17]:
import gradio as gr

In [18]:
all_pages = []
chunks = []
embeddings = None
index = None


In [19]:
#proccesing the pdf
def process_pdfs(files):

    global all_pages
    global chunks
    global embeddings
    global index

    if not files:

        return (
            "❌ Please upload at least one PDF.",
            gr.update(choices=[])
        )

    all_pages = []
    chunks = []
    embeddings = None
    index = None

    #here we extract the files

    for file in files:

        pdf_path = file

        pdf_pages = (
            extract_text_from_pdf(
                pdf_path
            )
        )

        all_pages.extend(
            pdf_pages
        )
         # Clean pages
    for page in all_pages:

        page["text"] = clean_text(
            page["text"]
        )

    # Create chunks
    chunks = create_chunks(
        all_pages,
        chunk_size=800,
        overlap=150
    )

    # Create embeddings
    chunk_texts = [
        chunk["text"]
        for chunk in chunks
    ]

    embeddings = (
        embedding_model.encode(
            chunk_texts,
            convert_to_numpy=True,
            show_progress_bar=False
        )
    )

    # Create FAISS index
    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(
        dimension
    )

    index.add(
        embeddings.astype(
            "float32"
        )
    )

    documents = sorted(
        set(
            chunk["document"]
            for chunk in chunks
        )
    )

    status = f"""
  PDF processing completed

  Documents: {len(documents)}
  Pages: {len(all_pages)}
  Chunks: {len(chunks)}
  Vectors: {index.ntotal}
"""

    return (
        status,
        gr.update(
            choices=documents,
            value=documents
        )
    )

In [20]:
# ASK DOCUMENTS
def run_rag(
    question,
    selected_documents,
    mode
):

    if not question.strip():

        return (
            "⚠️ Please enter a question.",
            ""
        )

    if index is None:

        return (
            "⚠️ Please upload and process "
            "your PDFs first.",
            ""
        )

    # If nothing selected,
    # search all documents.
    if not selected_documents:

        selected_documents = sorted(
            set(
                chunk["document"]
                for chunk in chunks
            )
        )

    answer, retrieved = (
        ask_documents(
            question,
            selected_documents,
            mode,
            top_k=5
        )
    )

    source_text = ""

    for source in retrieved:

        source_text += (
            f"📄 {source['document']} "
            f"| Page {source['page']}\n"
        )

    return (
        answer,
        source_text
    )


In [ ]:
import gradio as gr

# APPLICATION UI
with gr.Blocks(
    title="RAG Document Intelligence"
) as demo:

    gr.Markdown(
        """
# RAG Document Intelligence System

### Multi-PDF • Multiple Intelligence Modes • Citations

Upload your PDF documents, process them,
select the documents you want to use,
and ask questions.
"""
    )

    with gr.Row():
        with gr.Column(
            scale=1
        ):

            gr.Markdown(
                "##   Upload Documents"
            )

            pdf_files = gr.File(
                label="Upload PDF files",
                file_count="multiple",
                file_types=[".pdf"],
                type="filepath"
            )

            process_button = gr.Button(
                "⚙️ Process / Check PDFs",
                variant="primary"
            )

            status = gr.Markdown(
                "No documents processed yet."
            )

            document_selector = (
                gr.CheckboxGroup(
                    label="Select PDFs",
                    choices=[],
                    value=[]
                )
            )
        with gr.Column(
            scale=2
        ):

            gr.Markdown(
                "##   Document Intelligence"
            )

            mode = gr.Radio(
                choices=[
                    "Ask",
                    "Summarize",
                    "Compare",
                    "Extract",
                    "Research"
                ],
                value="Ask",
                label="Choose Intelligence"
            )

            question = gr.Textbox(
                label="Ask your documents",
                placeholder=(
                    "Example: What are the "
                    "main technologies discussed?"
                ),
                lines=4
            )

            ask_button = gr.Button(
                "️ Ask",
                variant="primary"
            )

            gr.Markdown(
                "##   Answer"
            )

            answer = gr.Markdown()

            gr.Markdown(
                "##   Sources"
            )

            sources = gr.Textbox(
                lines=8,
                interactive=False
            )
            # PROCESS BUTTON
    process_button.click(
        fn=process_pdfs,
        inputs=pdf_files,
        outputs=[
            status,
            document_selector
        ]
    )
    #Ask button
    ask_button.click(
        fn=run_rag,
        inputs=[
            question,
            document_selector,
            mode
        ],
        outputs=[
            answer,
            sources
        ]
    )
    #Enter key
    question.submit(
        fn=run_rag,
        inputs=[
            question,
            document_selector,
            mode
        ],
        outputs=[
            answer,
            sources
        ]
    )


demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e4a8addea496077509.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
